In [1]:
import pandas as pd
import os

In [73]:
# Function to calculate cost
def calculate_cost(row):
    # Base cost per km
    cost = row['Distance [km]'] * cost_factor_transportation
    
    # Add additional costs for Suez or Panama
    if pd.notna(row['Suez or Panama']):
        if 'Suez' in row['Suez or Panama']:
            cost += cost_factor_suez
        elif 'Panama' in row['Suez or Panama']:
            cost += cost_factor_panama
    
    return cost

# Function to create the new dataframe based on the "From" column for liquefaction cost
def create_LNG_liquefaction_df(df, factor):
    # Step 1: Create a new dataframe with unique "From" values and the corresponding "To" column
    unique_from_df = pd.DataFrame(df['From'].unique(), columns=['From'])
    
    # Step 2: Adjust the "From" column by removing "_LNG" and place the original "From" values in the "To" column
    unique_from_df['To'] = unique_from_df['From']
    unique_from_df['From'] = unique_from_df['From'].str.replace('_LNG', '')
    
    # Step 3: Add the "Cost" column with the value of cost_factor_liquification
    unique_from_df['Cost'] = factor
    
    return unique_from_df

# Function to create the new dataframe based on the "To" column for regasification cost
def create_LNG_regasification_df(df, factor):
    # Step 1: Create a new dataframe with unique "To" values and the corresponding "To" column
    unique_from_df = pd.DataFrame(df['To'].unique(), columns=['To'])
    
    # Step 2: Adjust the "To" column by removing "_LNG" and place the original "To" values in the "From" column
    unique_from_df['From'] = unique_from_df['To']
    unique_from_df['To'] = unique_from_df['To'].str.replace('_LNG', '')
    
    # Step 3: Add the "Cost" column with the value of cost_factor_liquification
    unique_from_df['Cost'] = factor
    
    return unique_from_df

#Define a function to multiply the distance by a cost factor for the pipelines
def pipeline_transport_cost(df, cost_factor):
    df["cost"] = df["distance [km]"] * cost_factor
    return df

# Function to create df_supply_demand_global
def create_supply_demand_df(df, supply=True):
    # Create the new dataframe with required columns
    df_supply_demand_global = pd.DataFrame({
        'Commodity': ['Methane'] * len(df),  # Set 'Methane' for all rows
        'Node': df['Country'] + ('_Prod' if supply else ''),  # Append '_Prod' if supply is True
        'Supply': df['GWh [2020]']  # Use 'GWh [2020]' for supply
    })
    
    return df_supply_demand_global

In [3]:
hydrogen_investment=False

cost_factor_transportation = 2
cost_factor_liquefaction = 2000
cost_factor_regasification = 2000
cost_factor_panama = 2000
cost_factor_suez = 2000

cost_factor_pipelines = 0.5

## Import Data

In [4]:
# Specify the path to your Excel file
input_file_path_1 = os.path.join('..', '..','01_data', '01_input_data', '02_processed')
excel_file_name = '\\Gas_Import_Russian_Invasion.xlsx'
# Specify the path to your Excel file
input_file_path_2 = os.path.join('..', '..','00_code_base', '07_data_prep')
distances_file_name = '\\distances.xlsx'

input_file_path_1  = input_file_path_1 + excel_file_name
full_input_path_1 = os.path.abspath(os.path.join(os.getcwd(), input_file_path_1))
input_file_path_2  = input_file_path_2 + distances_file_name
full_input_path_2 = os.path.abspath(os.path.join(os.getcwd(), input_file_path_2))

In [52]:
df_LNG_global = pd.read_excel(full_input_path_1, sheet_name='global_LNG_connections')
df_production_europe = pd.read_excel(full_input_path_1, sheet_name='nodes_gas_prod_europe_2020')
df_consumption_europe = pd.read_excel(full_input_path_1, sheet_name='nodes_demand_europe_2020')
df_production_global = pd.read_excel(full_input_path_1, sheet_name='nodes_gas_prod_world_2020')
df_consumption_global = pd.read_excel(full_input_path_1, sheet_name='nodes_demand_world_2020')

df_distances = pd.read_excel(full_input_path_2, sheet_name='distances')
#Drop the "Unnamed: 0" column in df_distances
df_distances.drop(columns=["Unnamed: 0"], inplace=True)

In [53]:
#adjust the data frames and remove unnecessary content
#gobal demand
df_consumption_global = df_consumption_global.iloc[:-3]
df_consumption_global = df_consumption_global.iloc[:, :-2]
#gobal production
df_production_global = df_production_global.iloc[:-3]
#europe demand
df_consumption_europe = df_consumption_europe.iloc[:-11]
#europe production
df_production_europe = df_production_europe.iloc[:-7]
df_production_europe = df_production_europe.iloc[:, :-7]

In [60]:
### Demand and Supply input sheet

In [66]:
#demand Europe
df_demand_europe = create_supply_demand_df(df_consumption_europe, supply=False)
#supply Europe
df_supply_europe = create_supply_demand_df(df_production_europe)

In [67]:
#demand Europe
df_demand_global = create_supply_demand_df(df_consumption_global, supply=False)
#supply global
df_supply_global = create_supply_demand_df(df_production_global)

In [71]:
# Combine the demand and supply data frames
df_all_supply_demand = pd.concat([df_demand_europe, df_supply_europe, df_demand_global, df_supply_global], ignore_index=True)

In [79]:
# Apply the function to each row and create a new column 'Cost per km'
df_LNG_global['Cost per Route'] = df_LNG_global.apply(calculate_cost, axis=1)

# Create a new dataframe with the relevant columns
df_cost_per_km = df_LNG_global[['From', 'To', 'Distance [km]', 'Suez or Panama', 'Cost per Route']]

# Display the new dataframe
df_cost_per_km

,From,To,Distance [km],Suez or Panama,Cost per Route
0,USA_LNG,SA_LNG,8566.540762,NaN,17133.081525
1,USA_LNG,AF_LNG,9968.279820,NaN,19936.559640
2,USA_LNG,CN_LNG,16405.657812,Panama,34811.315623
3,USA_LNG,JS_LNG,14912.186117,Panama,31824.372235
4,USA_LNG,IN_LNG,15398.208156,Suez,32796.416312
...,...,...,...,...,...
209,RU_LNG,EE_LNG,5244.853719,NaN,10489.707437
210,RU_LNG,FI_LNG,5244.853719,NaN,10489.707437
211,RU_LNG,DE_LNG,4393.510479,NaN,8787.020958
212,RU_LNG,IE_LNG,4951.953020,NaN,9903.906040


In [80]:
# Get LNG liquefaction cost df
LNG_liquefaction_df = create_LNG_liquefaction_df(df_LNG_global, cost_factor_liquefaction)
# Get LNG regasification cost df
LNG_regasification_df = create_LNG_regasification_df(df_LNG_global, cost_factor_regasification)

In [81]:
LNG_liquefaction_df

,From,To,Cost
0,USA,USA_LNG,2000
1,TT,TT_LNG,2000
2,ME,ME_LNG,2000
3,QA,QA_LNG,2000
4,AF,AF_LNG,2000
5,AU,AU_LNG,2000
6,RU,RU_LNG,2000
7,EG,EG_LNG,2000
8,IM,IM_LNG,2000


In [82]:
#Pipeline cost
#calculate European pipeline cost
df_result = pipeline_transport_cost(df_distances, cost_factor_pipelines)

In [83]:
#function to create the edges input with all capacities and cost
def create_edges_cap_cost_dataframe(df_cost_per_km, LNG_regasification_df, LNG_liquefaction_df):
    # Step 1: Collect unique From-To pairs from all three dataframes
    unique_pairs = set()

    # Extract From-To pairs from df_cost_per_km
    for _, row in df_cost_per_km.iterrows():
        unique_pairs.add((row['From'], row['To']))

    # Extract From-To pairs from LNG_regasification_df
    for _, row in LNG_regasification_df.iterrows():
        unique_pairs.add((row['From'], row['To']))

    # Extract From-To pairs from LNG_liquefaction_df
    for _, row in LNG_liquefaction_df.iterrows():
        unique_pairs.add((row['From'], row['To']))

    # Step 2: Create the new structured dataframe
    df_all_cost = pd.DataFrame(unique_pairs, columns=['Source', 'Destination'])

    # Step 3: Add required columns
    df_all_cost.insert(0, 'Commodity', 'Methane')  # Commodity column
    df_all_cost['initial_capacities'] = 9999  # Placeholder for now
    df_all_cost['max_capacities'] = 9999  # Placeholder for now

    # Step 4: Merge cost values from all input dataframes
    df_all_cost = df_all_cost.merge(
        df_cost_per_km[['From', 'To', 'Cost per Route']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To'])

    df_all_cost = df_all_cost.merge(
        LNG_regasification_df[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To'])

    df_all_cost = df_all_cost.merge(
        LNG_liquefaction_df[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To'])

    # Combine cost values (sum where both exist)
    df_all_cost['costs_edge'] = df_all_cost[['Cost per Route', 'Cost_x', 'Cost_y']].sum(axis=1, min_count=1)

    # Drop the temporary cost columns
    df_all_cost = df_all_cost.drop(columns=['Cost per Route', 'Cost_x', 'Cost_y'])

    # Step 5: Add empty columns for future values
    df_all_cost['new_build_cost'] = 1000000
    df_all_cost['conversion_cost'] = 0
    df_all_cost['conversion_capacity_factor'] = 1

    return df_all_cost

In [84]:
# Apply the create_edges_cap_cost_dataframe function to get edges input 
df_edges_cap_cost = create_edges_cap_cost_dataframe(df_cost_per_km, LNG_regasification_df, LNG_liquefaction_df)

# Display the result
df_edges_cap_cost

,Commodity,Source,Destination,initial_capacities,max_capacities,costs_edge,new_build_cost,conversion_cost,conversion_capacity_factor
0,Methane,EG_LNG,IE_LNG,9999,9999,9685.035188,1000000,0,1
1,Methane,ME_LNG,PL_LNG,9999,9999,24379.544588,1000000,0,1
2,Methane,RU_LNG,LT_LNG,9999,9999,9807.345370,1000000,0,1
3,Methane,AS_LNG,AS,9999,9999,2000.000000,1000000,0,1
4,Methane,RU_LNG,FI_LNG,9999,9999,10489.707437,1000000,0,1
...,...,...,...,...,...,...,...,...,...
242,Methane,ES_LNG,ES,9999,9999,2000.000000,1000000,0,1
243,Methane,RU_LNG,PL_LNG,9999,9999,9192.575772,1000000,0,1
244,Methane,ME_LNG,CN_LNG,9999,9999,19827.124214,1000000,0,1
245,Methane,RU_LNG,AS_LNG,9999,9999,40151.120667,1000000,0,1


In [75]:
def expand_with_hydrogen(df, hydrogen_investment):
    if hydrogen_investment:
        return df  # If investment is allowed, return the original dataframe unchanged

    # Select the first three columns (Commodity, Source, Destination)
    hydrogen_df = df[['Commodity', 'Source', 'Destination']].copy()

    # Replace Commodity with 'Hydrogen'
    hydrogen_df['Commodity'] = 'Hydrogen'

    # Fill other columns with 0
    for col in df.columns[3:]:  # Skip the first three columns
        hydrogen_df[col] = 0

    # Combine the original dataframe with the new hydrogen dataframe
    df_expanded = pd.concat([df, hydrogen_df], ignore_index=True)

    return df_expanded

In [77]:
def expand_with_hydrogen(df, hydrogen_investment):
    if hydrogen_investment:
        return df  # If investment is allowed, return the original dataframe unchanged

    # Check if the input dataframe follows the (Commodity, Source, Destination) structure
    if {'Commodity', 'Source', 'Destination'}.issubset(df.columns):
        # Copy relevant columns and create a hydrogen version
        hydrogen_df = df[['Commodity', 'Source', 'Destination']].copy()
        hydrogen_df['Commodity'] = 'Hydrogen'  # Replace Commodity with Hydrogen
        
    # If the input dataframe follows the (Commodity, Node, Supply) structure
    elif {'Commodity', 'Node', 'Supply'}.issubset(df.columns):
        # Copy relevant columns and create a hydrogen version
        hydrogen_df = df[['Commodity', 'Node']].copy()
        hydrogen_df['Commodity'] = 'Hydrogen'  # Replace Commodity with Hydrogen
    
    else:
        raise ValueError("Unexpected dataframe format. Must contain either ['Commodity', 'Source', 'Destination'] or ['Commodity', 'Node', 'Supply']")

    # Fill all other columns with 0
    for col in df.columns:
        if col not in hydrogen_df.columns:  # Skip the required columns
            hydrogen_df[col] = 0

    # Combine the original dataframe with the new hydrogen dataframe
    df_expanded = pd.concat([df, hydrogen_df], ignore_index=True)

    return df_expanded

In [85]:
# Example: Calling function with hydrogen_investment = False
df_edges_complete = expand_with_hydrogen(df_edges_cap_cost, hydrogen_investment)

# Display the result
df_edges_complete

,Commodity,Source,Destination,initial_capacities,max_capacities,costs_edge,new_build_cost,conversion_cost,conversion_capacity_factor
0,Methane,EG_LNG,IE_LNG,9999,9999,9685.035188,1000000,0,1
1,Methane,ME_LNG,PL_LNG,9999,9999,24379.544588,1000000,0,1
2,Methane,RU_LNG,LT_LNG,9999,9999,9807.345370,1000000,0,1
3,Methane,AS_LNG,AS,9999,9999,2000.000000,1000000,0,1
4,Methane,RU_LNG,FI_LNG,9999,9999,10489.707437,1000000,0,1
...,...,...,...,...,...,...,...,...,...
489,Hydrogen,ES_LNG,ES,0,0,0.000000,0,0,0
490,Hydrogen,RU_LNG,PL_LNG,0,0,0.000000,0,0,0
491,Hydrogen,ME_LNG,CN_LNG,0,0,0.000000,0,0,0
492,Hydrogen,RU_LNG,AS_LNG,0,0,0.000000,0,0,0


In [78]:
# Example: Calling function with hydrogen_investment = False
df_demand_supply_complete = expand_with_hydrogen(df_all_supply_demand, hydrogen_investment)

# Display the result
df_demand_supply_complete

,Commodity,Node,Supply
0,Methane,AL,-7717.933443
1,Methane,AT,-83311.215033
2,Methane,BE,-166072.218600
3,Methane,BA,-9469.790436
4,Methane,BG,-28534.750500
...,...,...,...
231,Hydrogen,IN_Prod,0.000000
232,Hydrogen,AS_Prod,0.000000
233,Hydrogen,CR_Prod,0.000000
234,Hydrogen,EG_Prod,0.000000


In [19]:
df_cost_per_km.to_excel("inputs.xlsx")